In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sympy.tensor import tensor
%matplotlib inline

In [3]:
# read in all the lines
words = open('src/names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [4]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi  = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos  = {i: s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# build the dataset
block_size = 3
X, Y = [], []
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator = g)
# F.one_hot(torch.tensor(5), num_classes = 27).float() @ C
emb = C[X]

In [7]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)
# roughly: emb @ W1 + b1, but there's no [32*3*2] @ [6*100]

# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1)
# torch.cat(torch.unbind(emb, 1), 1)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [8]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)
logits = h @ W2 + b2
counts = logits.exp()
probs  = counts / counts.sum(1, keepdim = True)

In [9]:
loss = -probs[torch.arange(32), Y].log().mean()
loss

tensor(14.7814)

##### cleaner
---

| training split | dev/validation split | test split |
|:---------------|:---------------------|:-----------|
| 80%            | 10%                  | 10%        |

---
Now we're splitting the words into these sets.

In [10]:
import random
def build_dataset(words):
    block_size = 3
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr,  Ytr  = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte,  Yte  = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [11]:
Xtr.shape, Ytr.shape

(torch.Size([182625, 3]), torch.Size([182625]))

In [12]:
# <<<<<------Make it cleaner.----->>>>>
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 10),   generator = g)
W1 = torch.randn((30, 200), generator = g)
b1 = torch.randn(200,       generator = g)
W2 = torch.randn((200, 27), generator = g)
b2 = torch.randn(27,        generator = g)
parameters = [C, W1, b1, W2, b2]

In [13]:
sum(p.nelement() for p in parameters)

11897

In [14]:
for p in parameters:
    p.requires_grad = True

In [15]:

for i in range(200000):
    # minibatch construct
    ix = torch.randint(0, Xtr.shape[0], (32,))

    # forward pass
    emb = C[Xtr[ix]]
    h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
    logits = h @ W2 + b2
    # counts = logits.exp()
    # probs = counts / counts.sum(1, keepdim = True)
    # loss = -probs[torch.arange(32), Y].log().mean()
    loss   = F.cross_entropy(logits, Ytr[ix])

    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    if i <= 50000:
        lr = 0.1
    elif i <= 150000:
        lr = 0.01
    else:
        lr = 0.001
    for p in parameters:
        p.data += -lr * p.grad

    # track stats
    # lri.append(lr.item())
    # lossi.append(loss.log10().item())
    # stepi.append(i)

print("loss:", loss.item())

loss: 2.32192063331604


In [16]:
emb = C[Xdev]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2
loss   = F.cross_entropy(logits, Ydev)
loss

tensor(2.1889, grad_fn=<NllLossBackward0>)

In [17]:
emb = C[Xte]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2
loss   = F.cross_entropy(logits, Yte)
loss

tensor(2.1915, grad_fn=<NllLossBackward0>)

In [18]:
emb = C[Xtr]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2
loss   = F.cross_entropy(logits, Ytr)
loss

tensor(2.1649, grad_fn=<NllLossBackward0>)

## Finally, sampling.

In [19]:
g = torch.Generator().manual_seed(2147483647 + 10)
for _ in range(20):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        h   = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        probs  = F.softmax(logits, dim = 1)
        ix     = torch.multinomial(probs, num_samples = 1, replacement = True, generator = g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    print(''.join(itos[i] for i in out))

carpavela.
jhquiffirleigelty.
halayan.
jazonte.
den.
rha.
kaeli.
nellara.
chaiiy.
kaleigh.
ham.
join.
quinn.
shoine.
livabi.
wanelo.
dearynix.
kaelynn.
demed.
edi.
